# BONSAI parser walkthrough

This notebook shows how to parse BONSAI's Balanced hybrid supply-use table (HSUT) and native b2 hybrid input-output table (HIOT). Monetary SUT exports are not supported.
The checked source directory is `20260615`, with reference year **2016**. Its name does not identify an immutable snapshot: individual files can be updated within it.


## What this notebook covers

- which local files the parser expects;
- how to read a Balanced HSUT or native b2 HIOT;
- how to inspect the parsed database;
- how to calculate electricity GHG multipliers with HSUT `fc` and HIOT `f`;
- how to compare HIOT results with saved BONSAI footprints while checking source versions, units and LCIA factors.

Jump to [Balanced HSUT](#Parse-the-HSUT) or [native b2 HIOT](#Parse-the-native-b2-HIOT).


## Relevant source pages

- [BONSAI project](https://bonsai.uno/)
- [BONSAI repository](https://gitlab.com/bonsamurais/bonsai)

Obtain the bundle from the project's shared data storage, under `_b Next version/Balanced` (HSUT) or `_b Next version/HIOT` (HIOT).
MARIO does not download BONSAI automatically.

## Main entry point

Use `mario.parse_bonsai(...)` with the original ZIP or its extracted directory. The default `table="SUT"` reads the Balanced HSUT; `table="IOT"` reads the native b2 HIOT demonstrated below.


## Key arguments and local layout

- `path`: a ZIP or extracted directory for the requested table;
- `table`: `"SUT"` (default) for the Balanced HSUT or `"IOT"` for the native b2 HIOT;
- `extensions`: `True` (default) includes the recalculated elementary flows; `False` reads only the economic tables.

For the Balanced HSUT, the parser reads these files:

```text
20260615/
├── classification.csv
├── hybrid_supply_w_characteristic_unit.csv
├── hybrid_use_w_characteristic_unit.csv
└── all_elementary_flows_AB_and_BB.csv
```

The last file is needed only when `extensions=True`. Other files in the bundle can be left in place.
See [mario.parse_bonsai](../../../api_document/mario.parse_bonsai.rst) for all arguments.

In [1]:
import mario

mario.set_log_verbosity("warning")

## Parse the HSUT

Pass the ZIP directly; no manual extraction is needed. You can also pass the extracted `20260615` directory.

The parser automatically selects `solve`, `umfpack`, `direct` and `partial` for this Python session, including subsequent calculations. Install the solver with `conda install -c conda-forge scikit-umfpack`. See the [API notes](../../../api_document/mario.parse_bonsai.rst) for details.

In [2]:
db = mario.parse_bonsai(
    path="/path/to/Balanced/20260615.zip",
)

db

/path/to/MARIO/mario/parsers/entrypoints.py:217: UserWarning: BONSAI: 362 elementary-flow rows have no classified activity/final demand and are omitted; examples: [('AT', 'A_MA_RECY'), ('BE', 'A_MA_RECY'), ('BE', 'A_SEWAGE'), ('BG', 'A_MA_LAND'), ('BG', 'A_MA_OTHE')]. Full rows are in the bonsai import report.
  matrices, indexes, units, layout = parse_bonsai_hsut(


name = BONSAI Balanced HSUT
table = SUT
tech_assumption = industry-based
scenarios = ['baseline']
Activity = from 365 to 1272 per region
Commodity = from 365 to 1335 per region
Factor of production = 1
Satellite account = 127
Consumption category = from 0 to 6 per region
Region = 50

## Inspect the parsed database

The result is a MARIO SUT database. Check its regions and the native commodity units before starting an analysis.

In [3]:
db.get_index("Region")

['AT',
 'AU',
 'BE',
 'BG',
 'BR',
 'CA',
 'CH',
 'CN',
 'CY',
 'CZ',
 'DE',
 'DK',
 'EE',
 'ES',
 'FI',
 'FR',
 'GB',
 'GR',
 'Global',
 'HR',
 'HU',
 'ID',
 'IE',
 'IN',
 'IT',
 'JP',
 'KR',
 'LT',
 'LU',
 'LV',
 'MT',
 'MX',
 'NL',
 'NO',
 'PL',
 'PT',
 'RO',
 'RU',
 'SE',
 'SI',
 'SK',
 'TR',
 'TW',
 'US',
 'WA',
 'WE',
 'WF',
 'WL',
 'WM',
 'ZA']

In [4]:
db.units["Commodity"].loc[["M_electricity"]]

,unit
M_electricity,TJ


## Electricity GHG footprint

Characterize the greenhouse gases using the built-in BONSAI profile (AR6, GWP100), then retain the aggregated GHG account. Fossil CO2, fossil and non-fossil methane, and N2O are included; biogenic CO2 stays separate. See the [GHG API](../../../api_document/mario.Database.calc_ghg.rst) for the factors.

Keeping only GHG before calculating `fc` limits the solve to one environmental row.

In [5]:
db.calc_ghg()
ghg = "GHG AR6 GWP-100"
db.keep_extensions([ghg])

db.units["Satellite account"]

,unit
Item,
GHG AR6 GWP-100,tonnes


`fc` contains direct and upstream emissions per unit of each regional commodity. For the GHG row and `M_electricity`, its units are **tonnes CO2eq/TJ**.

The four commodity coefficients affected by singular components in this release are reported as `NaN`, with a warning. The electricity markets selected below are independent of those components.

In [6]:
fc = db.fc
fc.shape

/path/to/MARIO/mario/compute/linear_solvers.py:220: PartialResultWarning: Partial linear-system result: 4 of 53400 coefficients per right-hand side are NaN because they belong to, or depend on, singular production components. Singular components: [{'nodes': [('TW', 'Commodity', 'C_Beany'), ('TW', 'Commodity', 'M_Beany')], 'count': 2}, {'nodes': [('TW', 'Commodity', 'C_Eggs_ggs'), ('TW', 'Commodity', 'M_Eggs_ggs')], 'count': 2}]. Only independent equations are solved; the full system may be inconsistent or have nonunique solutions. Source matrices are unchanged. Totals including undefined coefficients must propagate NaN (use skipna=False for pandas reductions).
  solved = np.asarray(factor.solve(rhs_values), dtype=np.float64)


(1, 53400)

Select `M_electricity` across regions. Each result is the GHG footprint of **1 kWh supplied by that regional electricity market**, including its upstream supply chain.

**Unit conversion:** multiplying by **3.6** converts tonnes CO2eq/TJ into g CO2eq/kWh, since 1 tonne = 10⁶ g and 1 kWh = 3.6 × 10⁻⁶ TJ.

In [7]:
electricity_ghg = fc.loc[
    ghg, (slice(None), slice(None), "M_electricity")
] * 3.6

electricity_ghg.droplevel([1, 2]).to_frame("g CO2eq/kWh")

,g CO2eq/kWh
Region,
AT,244.583914
AU,735.553038
BE,233.659069
BG,545.943557
BR,123.240659
CA,155.694559
CH,177.398858
CN,859.220591
CY,1745.262889


## Notes

- Activity and commodity classifications can differ between regions. This is a [special case of the index API](../../../get_started/nomenclature.rst); see [Basic inspections](../../../user_guide/inspection/basic_inspections.ipynb) for the inspection workflow.
- Value-added accounts are not available in these inputs; MARIO uses zero placeholders.
- Satellite labels use `name | compartment`; IDs disambiguate homonyms only. Units and native identities remain in `db.units["Satellite account"]` and `db.info["bonsai"]["satellites"]`. `db.calc_ghg()` auto-detects the BONSAI profile; see the [GHG API](../../../api_document/mario.Database.calc_ghg.rst) for its factors and scope.
- The default `calc_all=False` keeps the imported blocks sparse. Full tables are large: reduce the model before requesting full inverse matrices.
- Import diagnostics are available in `db.info["bonsai"]`. Unmatched elementary-flow rows generate a warning by default; use `unmatched_extensions="raise"` to reject them.

## Parse the native b2 HIOT

Use `table="IOT"` with the local synchronized `HIOT/20260615` directory or a ZIP containing one release. Required files are `extended_size_hiot.b2.pkl`, `final_demand.b2.csv` and `total_elementary_flow_w_iLUC.csv`. Only use a trusted BONSAI pickle export. The parser imports the final Stone/iLUC/capital model; it does not reconstruct it from the HSUT.

The reference year is inferred from elementary-flow `time` values; pass `year=2016` if those values are unavailable. Original product codes and region-specific Sector lists remain present. The activity-to-reference-product mapping is in `db_iot.info["bonsai"]["reference_products"]`.

The native dimensional matrix `T` is represented by `X = diag(T)` and `Z = diag(X) - T`. This retains substitution credits and source diagonal normalization, independently of source production imbalance. Gross reference supply and self consumption cannot be separated from this matrix alone. HIOT per-unit multipliers use `f`, while the HSUT examples above use `fc`.

Missing final-demand products are reported with a warning and full source rows in `unmatched_final_demand`; use `unmatched_final_demand="raise"` to reject them. `unmatched_extensions` controls the analogous elementary-flow policy. No source balance is repaired, and no precomputed inverse or footprint is loaded.

For `Q = diag(X)`, `z = Z Q^-1` and `e = E Q^-1`, this representation gives `T = (I-z)Q` and therefore **`f = e (I-z)^-1 = E T^-1`**. The solve avoids constructing the full inverse.


In [2]:
from pathlib import Path
import mario

hiot_path = Path("/path/to/HIOT/20260615")
db_iot = mario.parse_bonsai(path=hiot_path, table="IOT")

sectors_it = db_iot.get_index("Sector", region="IT")
print(f"Reference year: {db_iot.meta.year}")
print(f"Native production nodes: {db_iot.X.shape[0]:,}")
print(f"Sectors in Italy: {len(sectors_it)}")
print("First Italian sectors:", sectors_it[:8])

ghg = "GHG AR6 GWP-100"
db_iot.calc_ghg()


Reference year: 2016
Native production nodes: 53,789
Sectors in Italy: 1230
First Italian sectors: ['C_ACCOM|C_RESTAU', 'C_ACT_HOUS', 'C_ADVERT', 'C_ALUM_01_RES', 'C_ALUW', 'C_ALU_sort', 'C_AL_INCI', 'C_AL_LAND']


## HIOT GHG characterization

`calc_ghg()` applies MARIO's built-in AR6 GWP100 profile. It does not load BONSAI's LCIA workbook. The local `LCIA.xlsx` checked on **20 October 2026** uses the same factors for fossil CO2, fossil/non-fossil methane and N2O, and additionally uses **0.00772** for `Accelerated CO2, Air`.

The following mapping reproduces the nonzero GWP100 factors in that inspected workbook. Verify it against the LCIA file for your source export. UUIDs in the workbook can be matched to readable labels through `db_iot.info["bonsai"]["satellites"]`.

Create both characterized rows before filtering the elementary flows. Keeping the two aggregates limits the HIOT solve to two right-hand sides.


In [3]:
bonsai_gwp100 = {
    "Carbon dioxide, fossil | Air": 1.0,
    "Methane, fossil | Air": 29.8,
    "Methane, non-fossil | Air": 27.0,
    "Dinitrogen monoxide | Air": 273.0,
    "Accelerated CO2, Air | Air": 0.00772,
}
bonsai_ghg = "GHG BONSAI GWP100"
db_iot.calc_ghg(gwp=bonsai_gwp100, label=bonsai_ghg)
db_iot.keep_extensions([ghg, bonsai_ghg])
f_iot = db_iot.f


## HIOT electricity multipliers

The native `M_electricity` unit is TJ. Both GHG rows are in tonnes CO2eq/TJ; multiplying by **3.6** gives **g CO2eq/kWh**. The regional index below contains only existing electricity nodes.


In [4]:
db_iot.units["Sector"].loc[["M_electricity"]]

electricity_iot = f_iot.loc[
    [ghg, bonsai_ghg], (slice(None), slice(None), "M_electricity")
].T * 3.6
electricity_iot.index = electricity_iot.index.droplevel([1, 2])
electricity_iot.index.name = "region"
electricity_iot


Item,GHG AR6 GWP-100,GHG BONSAI GWP100
region,,
AT,321.275249,321.304841
AU,990.324681,990.353107
BE,455.780431,455.910551
BG,614.933613,615.027795
BR,246.896357,247.028873
CA,264.256311,264.271051
CH,395.860139,395.951162
CN,1190.087963,1190.189869
CY,1550.872109,1550.972976


## Compare with saved BONSAI footprints

Use `ghg_for_all.b2.GWP100.csv` for the same source snapshot and native units. For the single product below, the product/unit pair is selected explicitly and the remaining index is the region. Use `(region, product, unit)` when comparing several products.

An outer join retains missing nodes. Percentage differences are undefined for a zero or missing saved value; signed or near-zero footprints need additional care when interpreting percentages. This example reads the saved GHG footprint only, without loading a precomputed inverse.


In [5]:
import pandas as pd

saved_ghg = pd.read_csv(
    hiot_path / "ghg_for_all.b2.GWP100.csv", index_col=[0, 1]
)
saved_electricity = saved_ghg.loc[("M_electricity", "TJ")] * 3.6
comparison = electricity_iot.join(
    saved_electricity.rename("BONSAI saved"), how="outer"
)
denominator = comparison["BONSAI saved"].replace(0, float("nan"))
comparison["MARIO difference (%)"] = (comparison[ghg] / denominator - 1) * 100
comparison["BONSAI CF difference (%)"] = (
    comparison[bonsai_ghg] / denominator - 1
) * 100
comparison


,GHG AR6 GWP-100,GHG BONSAI GWP100,BONSAI saved,MARIO difference (%),BONSAI CF difference (%)
AT,321.275249,321.304841,318.242340,0.953019,0.962317
AU,990.324681,990.353107,990.023337,0.030438,0.033309
BE,455.780431,455.910551,487.086707,-6.427249,-6.400535
BG,614.933613,615.027795,639.065378,-3.776103,-3.761365
BR,246.896357,247.028873,242.017773,2.015796,2.070551
CA,264.256311,264.271051,386.296251,-31.592318,-31.588502
CH,395.860139,395.951162,383.946835,3.102852,3.126560
CN,1190.087963,1190.189869,1124.752599,5.808865,5.817926
CY,1550.872109,1550.972976,1501.944945,3.257587,3.264303
CZ,850.854750,850.919611,860.773911,-1.152354,-1.144819


## Dated local verification

The local verification on **20 October 2026** imported **53,789 production nodes** and preserved the native matrix exactly. The full calculation of 129 environmental rows satisfied `f @ T = E` with a maximum row-wise relative residual of **1.60e-14**.

The compared files have different modification dates: the matrix on 20 October, elementary flows on 19 October, and the saved GWP100 footprint on 15 October. The native model also has 1,472 nodes absent from the saved footprints; 78 saved nodes are absent from the native model. No separate HIOT snapshot from 15 October was found in the inspected local HIOT/version directories. **Modification dates are not release identifiers**, and these files do not establish an equivalence benchmark using matched inputs.

For electricity, in g CO2eq/kWh:

| Region | MARIO AR6 GWP100 | BONSAI saved | Difference |
|---|---:|---:|---:|
| Italy | 449.74 | 435.65 | +3.23% |
| Germany | 613.36 | 606.56 | +1.12% |
| France | 440.16 | 411.28 | +7.02% |
| Canada | 264.26 | 386.30 | -31.59% |
| Indonesia | 769.59 | 592.77 | +29.83% |

Across 48 comparable electricity markets the median absolute difference is **3.11%**, and 34 markets are within 5%. Across 50,846 comparable nonzero saved product footprints it is **5.85%**. Using the checked BONSAI LCIA factors changes Italy's result to **449.79 g CO2eq/kWh**, leaving a **+3.25%** difference from the saved footprint.

The [verification report](hiot_validation_20261020.json) records checksums, source modification times and residuals; the [electricity CSV](electricity_comparison_20261020.csv) contains all 48 compared markets. For future checks, archive the matrix, elementary flows, final demand, LCIA workbook and saved footprint together. A small residual verifies the imported linear system; agreement with a saved footprint additionally requires matching inputs and characterization.
